# Two-stage XGBoost: predictions across the scheduling week

Use **`engineered_scheduling` (262 features)**. Your reported public results were 0.99053 for scheduling alone, 0.99046 for hydrology, and **0.99107 for the 50/50 baseline+deeper scheduling ensemble**. This notebook retains that winning ensemble as stage 1 and tests a small temporal correction model as stage 2. Improvement is not established until you run validation and submit predictions.

Upload the four matching scheduling files together in Colab: `hourly_train.csv`, `hourly_test.csv`, `cases.csv`, and `meta.json`. They can be directly in `/content`. No original inflow CSV or new feature-engineering run is needed. CUDA is the default; CuPy is not required.

Stage 2 uses stage-1 probabilities for all units, nearby hours of the same unit, weekly summaries, and a few existing scheduling inputs. Its initial prediction is the stage-1 probability, using its **log-odds as XGBoost's `base_margin`**. Small trees learn corrections rather than starting from scratch. The same margin is supplied at prediction time. [XGBoost documentation](https://xgboost.readthedocs.io/en/stable/prediction.html#base-margin)

All prediction windows stay inside one 168-hour case. Future-hour **predictions** are allowed because the complete week's scheduling inputs are supplied; future-hour **labels** never become features. Cases from different start dates are never averaged together.

In [ ]:
# Colab only, if XGBoost is missing or older than 2.0; restart after changing packages.
# %pip install -q "xgboost>=2.0"
# No installation commands are needed during offline exam reproduction.

In [ ]:
from pathlib import Path
import gc
import json
import time
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits
from xgboost import DMatrix, XGBClassifier

DATA_PATH = None  # Auto-detect, or set Path("/content") for flat Colab uploads.
DEVICE = "cuda"
N_THREADS = 4
HORIZON = 168
META_TRAIN_YEARS = [2019, 2020]
SELECTION_YEAR = 2021
CONFIRMATION_YEAR = 2022
RUN_SUBMISSION = False
MIN_GAIN = 0.0001  # Practical adoption margin, not statistical significance.
BLEND_WEIGHTS = [0.0, 0.25, 0.5, 0.75, 1.0]  # Weight assigned to stage 2.
SUBMISSION_PATH = Path("submission_xgboost_temporal.csv")

COMMON = dict(objective="binary:logistic", eval_metric="auc", tree_method="hist",
              device=DEVICE, random_state=1337, n_jobs=N_THREADS)
BASELINE = dict(**COMMON, n_estimators=300, learning_rate=0.05, max_depth=5,
                min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
                reg_lambda=1.0, reg_alpha=0.0)
STAGE1_PARAMS = {
    "baseline": BASELINE,
    "deeper": {**BASELINE, "n_estimators": 600, "learning_rate": 0.035,
               "max_depth": 7, "min_child_weight": 10, "reg_lambda": 5.0},
}
STAGE2_PARAMS = dict(**COMMON, n_estimators=120, learning_rate=0.03, max_depth=3,
                     min_child_weight=30, subsample=1.0, colsample_bytree=1.0,
                     reg_lambda=10.0, reg_alpha=0.1)
if (not META_TRAIN_YEARS or META_TRAIN_YEARS != sorted(set(META_TRAIN_YEARS))
        or max(META_TRAIN_YEARS) >= SELECTION_YEAR or SELECTION_YEAR >= CONFIRMATION_YEAR):
    raise ValueError("Use increasing, distinct meta-training years, then selection, then confirmation.")
if 0.0 not in BLEND_WEIGHTS or any(w < 0 or w > 1 for w in BLEND_WEIGHTS):
    raise ValueError("Blend weights must be in [0, 1] and include 0 for the original ensemble.")

In [ ]:
# Test XGBoost itself before loading data. NumPy inputs do not require CuPy.
if DEVICE.startswith("cuda"):
    import shutil
    import subprocess
    import xgboost
    print("XGBoost:", xgboost.__version__)
    if shutil.which("nvidia-smi"):
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        print(result.stdout or result.stderr)
    else:
        print("nvidia-smi unavailable. Check that Colab has a GPU runtime selected.")
    try:
        probe_X = np.array([[0.0], [1.0], [2.0], [3.0]], dtype=np.float32)
        probe = XGBClassifier(device=DEVICE, tree_method="hist", n_estimators=1, max_depth=1)
        probe.fit(probe_X, np.array([0, 0, 1, 1]))
        booster = probe.get_booster()
        actual_device = json.loads(booster.save_config())["learner"]["generic_param"]["device"]
        if not actual_device.startswith("cuda"):
            raise RuntimeError(f"XGBoost selected {actual_device} instead of CUDA.")
        booster.predict(DMatrix(probe_X))
        print(f"XGBoost CUDA training and prediction verified: {actual_device}")
        del probe, booster, probe_X
    except Exception as error:
        raise RuntimeError(
            "XGBoost could not use CUDA. Select a Colab GPU runtime and restart the session "
            "after package changes. If this persists, share the nvidia-smi output and XGBoost "
            "version printed above to check driver compatibility. No full training has started. "
            f"Original error: {error}"
        ) from error
elif DEVICE != "cpu":
    raise ValueError("Set DEVICE to 'cuda' or 'cpu'.")

In [ ]:
if DATA_PATH is None:
    folders = (Path("engineered_scheduling"), Path("../engineered_scheduling"), Path("."))
    DATA_PATH = next((p for p in folders if (p / "hourly_train.csv").is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Upload the scheduling CSVs and meta.json together, or set DATA_PATH.")
DATA_PATH = Path(DATA_PATH).resolve()
print(f"Dataset: {DATA_PATH}")
meta = json.loads((DATA_PATH / "meta.json").read_text())
UNITS, TARGETS = meta["units"], meta["y_cols"]
FEATS = [c for c in meta["hourly_feature_cols"] + meta["static_feature_cols"]
         if not c.startswith("dvol_")]
if meta.get("feature_engineering", {}).get("variant") != "scheduling_context":
    raise ValueError("Use the matching engineered_scheduling CSVs and meta.json for this comparison.")
if (TARGETS != [f"y_{u}" for u in UNITS] or len(FEATS) != len(set(FEATS))
        or set(FEATS).intersection(TARGETS + ["Run No", "starttime", "split", "fold"])):
    raise ValueError("Feature/target metadata is inconsistent.")

cases = pd.read_csv((DATA_PATH / "cases.csv"), dtype={c: np.float32 for c in meta["static_feature_cols"]})
cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True)
if cases["Run No"].isna().any() or cases["Run No"].duplicated().any() or cases["starttime"].isna().any():
    raise ValueError("cases.csv must have unique Run No values and valid start times.")


def load_hourly(path, split):
    targets = TARGETS if split == "train" else []
    dtypes = {c: np.float32 for c in meta["hourly_feature_cols"] + targets}
    dtypes.update({"Run No": np.int64, "t": np.int16})
    hourly = pd.read_csv(path, dtype=dtypes)
    expected_runs = set(cases.loc[cases["split"].eq(split), "Run No"])
    if set(hourly["Run No"]) != expected_runs or hourly.duplicated(["Run No", "t"]).any():
        raise ValueError(f"{path.name}: missing/unexpected cases or duplicate (Run No, t) rows.")
    rows = hourly.merge(cases, on="Run No", how="left", validate="many_to_one")
    rows = rows.sort_values(["Run No", "t"]).reset_index(drop=True)
    if (not rows.groupby("Run No").size().eq(HORIZON).all()
            or not np.array_equal(rows["t"], np.tile(np.arange(HORIZON), len(expected_runs)))):
        raise ValueError(f"{path.name}: each case must have exactly the 168 hours t=0..167.")
    if not np.isfinite(rows[FEATS].to_numpy(dtype=np.float32)).all():
        raise ValueError(f"{path.name}: missing or infinite feature values.")
    return rows


train = load_hourly((DATA_PATH / "hourly_train.csv"), "train")
if not np.isin(train[TARGETS].to_numpy(), [0, 1]).all():
    raise ValueError("Training labels must be 0 or 1.")
row_info = train[["Run No", "t", "starttime"]].copy()
X = train[FEATS].to_numpy(dtype=np.float32)
y = train[TARGETS].to_numpy(dtype=np.uint8)
del train  # Keep the arrays and identifiers; avoid retaining a second feature matrix.
print(f"Loaded {row_info['Run No'].nunique():,} cases | {len(X):,} hourly rows | {len(FEATS)} features | {len(UNITS)} units")

OOF_CACHE = {}  # In-memory only; loading another dataset resets cached predictions.

## Chronological training plan

| Step | Stage-2 training examples | Evaluation |
|---|---|---|
| Choose correction weight | Out-of-fold predictions for 2019 and 2020 | 2021 |
| Confirm the frozen weight | Out-of-fold predictions for 2019–2021 | 2022 |
| Final refit | Out-of-fold predictions for 2019–2022 | Unlabeled Kaggle cases |

For each out-of-fold year, stage 1 trains only on cases **ending before that year**. Stage-2 training cases must also end before their evaluation year; late-December overlap cases are removed at both levels. Earlier data still trains stage 1 but has no stage-2 labels unless its year is listed above. Yearly first-stage predictions are cached in RAM and reused when rerunning the correction or blend cells. Reloading data clears the cache; changing stage-1 parameters creates new cache entries.

Default validation costs **112 stage-1 fits + 28 small stage-2 fits**. Final submission adds 28 stage-1 fits and, if enabled by the selected weight, 14 small stage-2 fits. All are sequential on one GPU. Set `META_TRAIN_YEARS = [2020]` before starting for a shorter first experiment, reducing validation to 84 + 28 fits. No full CPU run is required here.

2022 has already influenced earlier project decisions, so this is a development comparison, not an untouched test score. Blend selection always includes the unchanged stage-1 ensemble.

In [ ]:
def auc(labels, probabilities):
    return roc_auc_score(labels.ravel(), probabilities.ravel()) if np.unique(labels).size > 1 else np.nan


def year_indices(year):
    start = pd.Timestamp(f"{year}-01-01", tz="UTC")
    end = pd.Timestamp(f"{year + 1}-01-01", tz="UTC")
    dates = row_info["starttime"]
    fit = np.flatnonzero((dates + pd.Timedelta(hours=HORIZON) <= start).to_numpy())
    valid = np.flatnonzero(((dates >= start) & (dates < end)).to_numpy())
    if not len(fit) or not len(valid):
        raise ValueError(f"No training or validation cases for {year}.")
    return fit, valid


def fit_first_stage(X_fit, y_fit, X_predict):
    result = np.zeros((len(X_predict), len(UNITS)), dtype=np.float32)
    matrix = DMatrix(X_predict, nthread=N_THREADS)
    with threadpool_limits(limits=N_THREADS):
        for name, params in STAGE1_PARAMS.items():
            started = time.perf_counter()
            for j, unit in enumerate(UNITS):
                classes = np.unique(y_fit[:, j])
                if len(classes) == 1:
                    probability = classes[0]
                else:
                    model = XGBClassifier(**params)
                    model.fit(X_fit, y_fit[:, j])
                    probability = model.get_booster().predict(matrix)
                    del model
                result[:, j] += probability / len(STAGE1_PARAMS)
                print(f"  {name}: {j + 1}/{len(UNITS)} {unit}", flush=True)
            print(f"{name}: {time.perf_counter() - started:.1f}s", flush=True)
    del matrix
    gc.collect()
    return result


def get_oof(year):
    key = (year, HORIZON, json.dumps(STAGE1_PARAMS, sort_keys=True))
    if key not in OOF_CACHE:
        fit, valid = year_indices(year)
        print(f"OOF {year}: {len(fit) // HORIZON:,} training cases; {len(valid) // HORIZON:,} prediction cases")
        OOF_CACHE[key] = (valid, fit_first_stage(X[fit], y[fit], X[valid]))
    else:
        print(f"Reusing cached {year} predictions.")
    return OOF_CACHE[key]


def stack_oof(years, before=None):
    records = [get_oof(year) for year in years]
    indices = np.concatenate([r[0] for r in records])
    probabilities = np.concatenate([r[1] for r in records])
    if before is not None:
        boundary = pd.Timestamp(f"{before}-01-01", tz="UTC")
        keep = (row_info.iloc[indices]["starttime"] + pd.Timedelta(hours=HORIZON) <= boundary).to_numpy()
        indices, probabilities = indices[keep], probabilities[keep]
    if not len(indices):
        raise ValueError("No stage-2 training cases remain after the overlap gap.")
    print(f"Stage-2 training: {len(indices) // HORIZON:,} out-of-fold cases")
    return indices, probabilities

## Small correction model

For each unit, stage 2 receives the current probabilities of all 14 units; its own predictions at ±1, ±3, ±6, ±12 and ±24 hours; past/next 6- and 24-hour means; weekly mean, standard deviation, minimum and maximum; and selected existing price, storage and time inputs. Boundary shifts repeat the edge of the same case; mean windows are truncated at case edges.

No true ON/OFF value is supplied to the feature builder. Fixed 120-tree, depth-3 models limit complexity. The original ensemble supplies the starting log-odds; a weight of zero returns its probabilities exactly. There is no hard thresholding, forced generator order, or smoothing across separate cases.

In [ ]:
def window_mean(values, width, forward):
    t = np.arange(HORIZON)
    left = t if forward else np.maximum(0, t - width + 1)
    right = np.minimum(HORIZON, t + width) if forward else t + 1
    cumulative = np.pad(np.cumsum(values, axis=1, dtype=np.float64), ((0, 0), (1, 0)))
    return (cumulative[:, right] - cumulative[:, left]) / (right - left)


def second_stage_features(features, probabilities, unit_index):
    if len(features) != len(probabilities) or len(features) % HORIZON:
        raise ValueError("Stage-2 inputs must contain complete, aligned 168-hour cases.")
    q = probabilities[:, unit_index].reshape(-1, HORIZON)
    t = np.arange(HORIZON)
    columns = [probabilities]  # Current prediction of every unit; no labels.
    for lag in (-24, -12, -6, -3, -1, 1, 3, 6, 12, 24):
        columns.append(q[:, np.clip(t + lag, 0, HORIZON - 1)].ravel())
    for width in (6, 24):
        for forward in (False, True):
            columns.append(window_mean(q, width, forward).ravel())
    for values in (q.mean(1), q.std(1), q.min(1), q.max(1)):
        columns.append(np.repeat(values, HORIZON))
    plant = meta["unit_plant"][UNITS[unit_index]]
    context = ["t", "price", "price_rank_week", "price_rank_day", "price_z",
               "price_diff_prev", "price_diff_next", f"price_minus_wvup_{plant}",
               f"fe_relative_price_spread_{plant}", f"fe_initial_fill_up_{plant}",
               f"fe_initial_fill_down_{plant}"]
    columns.extend(features[:, FEATS.index(c)] for c in context if c in FEATS)
    return np.column_stack(columns).astype(np.float32)


def log_odds(probabilities):
    p = np.clip(probabilities, 1e-5, 1 - 1e-5)
    return (np.log(p) - np.log1p(-p)).astype(np.float32)


def fit_second_stage(X_fit, y_fit, base_fit, X_predict, base_predict):
    result = np.empty_like(base_predict)
    with threadpool_limits(limits=N_THREADS):
        for j, unit in enumerate(UNITS):
            classes = np.unique(y_fit[:, j])
            if len(classes) == 1:
                result[:, j] = classes[0]
            else:
                z_fit = second_stage_features(X_fit, base_fit, j)
                z_predict = second_stage_features(X_predict, base_predict, j)
                model = XGBClassifier(**STAGE2_PARAMS)
                model.fit(z_fit, y_fit[:, j], base_margin=log_odds(base_fit[:, j]))
                matrix = DMatrix(z_predict, base_margin=log_odds(base_predict[:, j]), nthread=N_THREADS)
                result[:, j] = model.get_booster().predict(matrix)
                del z_fit, z_predict, matrix, model
            print(f"  Stage 2: {j + 1}/{len(UNITS)} {unit}", flush=True)
    gc.collect()
    return result


def mix(base, refined, weight):
    return (1 - weight) * base + weight * refined

## Select the correction weight on 2021

The 2021 labels are used only to compare the five fixed weights, not to fit either stage for this evaluation. Cached first-stage predictions make later stage-2 experiments cheaper. After changing a model configuration, rerun selection, confirmation and submission in order.

Selection also excludes cases whose horizons overlap the start of the confirmation year (359 cases in the supplied 2021 data), so blend-weight selection does not reuse those overlapping calendar hours.

In [ ]:
meta_indices, meta_probabilities = stack_oof(META_TRAIN_YEARS, before=SELECTION_YEAR)
selection_indices, selection_base = get_oof(SELECTION_YEAR)
# Also separate weight-selection labels from the following confirmation horizon.
confirmation_start = pd.Timestamp(f"{CONFIRMATION_YEAR}-01-01", tz="UTC")
keep = (row_info.iloc[selection_indices]["starttime"] + pd.Timedelta(hours=HORIZON) <= confirmation_start).to_numpy()
selection_indices, selection_base = selection_indices[keep], selection_base[keep]
if not len(selection_indices):
    raise ValueError("No selection cases remain before the confirmation gap.")
selection_refined = fit_second_stage(X[meta_indices], y[meta_indices], meta_probabilities,
                                     X[selection_indices], selection_base)
selection_results = pd.DataFrame([
    {"stage2_weight": w, "micro_auc": auc(y[selection_indices], mix(selection_base, selection_refined, w))}
    for w in BLEND_WEIGHTS
]).set_index("stage2_weight")
selection_results["gain_over_stage1"] = selection_results.micro_auc - selection_results.loc[0.0, "micro_auc"]
selection_results = selection_results.sort_values("micro_auc", ascending=False, kind="stable")
selected_weight = float(selection_results.index[0])
if selection_results.loc[selected_weight, "gain_over_stage1"] < MIN_GAIN:
    selected_weight = 0.0
recommended_weight = 0.0  # Reset confirmation state when selection is rerun.
confirmation_results = None
print(f"Selected stage-2 weight: {selected_weight:.2f}")
display(selection_results.round(6))
del meta_indices, meta_probabilities

## Confirm on 2022

Refit the same small second stage on earlier out-of-fold cases, including 2021. Its blend weight remains fixed. The unchanged stage-1 ensemble is the reference. Keep stage 1 if the chosen correction fails to improve by `MIN_GAIN`. If selection chose zero, this cell skips stage-2 fitting for confirmation.

In [ ]:
recommended_weight = 0.0
confirmation_indices, confirmation_base = get_oof(CONFIRMATION_YEAR)
confirmation_refined = confirmation_base
if selected_weight > 0:
    meta_indices, meta_probabilities = stack_oof(META_TRAIN_YEARS + [SELECTION_YEAR], before=CONFIRMATION_YEAR)
    confirmation_refined = fit_second_stage(X[meta_indices], y[meta_indices], meta_probabilities,
                                            X[confirmation_indices], confirmation_base)
    del meta_indices, meta_probabilities
confirmation_selected = mix(confirmation_base, confirmation_refined, selected_weight)
baseline_auc = auc(y[confirmation_indices], confirmation_base)
selected_auc = auc(y[confirmation_indices], confirmation_selected)
if selected_weight > 0 and selected_auc - baseline_auc >= MIN_GAIN:
    recommended_weight = selected_weight
confirmation_results = pd.DataFrame({
    "micro_auc": [baseline_auc, selected_auc],
    "gain_over_stage1": [0.0, selected_auc - baseline_auc],
}, index=["stage1_ensemble", "selected_two_stage"])
display(confirmation_results.round(6))
unit_results = pd.DataFrame({
    "stage1": [auc(y[confirmation_indices, j], confirmation_base[:, j]) for j in range(len(UNITS))],
    "selected": [auc(y[confirmation_indices, j], confirmation_selected[:, j]) for j in range(len(UNITS))],
}, index=UNITS)
display(unit_results.round(5))
print(f"Recommended final stage-2 weight: {recommended_weight:.2f}")
print("Weight 0 retains the original baseline+deeper ensemble.")

## Optional Kaggle CSV

Set `RUN_SUBMISSION = True` in the next cell. `FINAL_STAGE2_WEIGHT = None` uses the recommendation; an explicit number in [0, 1] overrides it for a deliberate experiment. Final stage 1 trains on **all labeled cases**. Final stage 2 trains on the cached, out-of-fold predictions for 2019–2022, not on stage-1 predictions of its own training data. The oldest years remain stage-1 training only. No test labels or observed future storage are used.

Only `submission_xgboost_temporal.csv` is exported. All cached predictions and result tables stay in memory. Restarting the runtime loses that cache. The full-data stage 1 has more training data than the fold models; this is the usual stacking distribution change and is one reason improvement must be validated.

**Exam delivery:** This is a development notebook using the scheduling CSVs. Before submitting the short notebook to Canvas, include the required raw-data preprocessing and scheduling feature code in the same notebook, pin the software versions in the allowed configuration file, and remove development-only result comparisons. Reproducible out-of-fold generation is still necessary to train stage 2; it cannot rely on saved predictions or model files from this session. Run offline, document the GPU if used, and check the assignment's time limit.

In [ ]:
# RUN_SUBMISSION = True
FINAL_STAGE2_WEIGHT = None  # None = recommendation; e.g. 0.5 = explicit experiment.
if RUN_SUBMISSION:
    final_weight = recommended_weight if FINAL_STAGE2_WEIGHT is None else float(FINAL_STAGE2_WEIGHT)
    if not 0 <= final_weight <= 1:
        raise ValueError("FINAL_STAGE2_WEIGHT must be between zero and one.")
    print(f"Final stage-2 weight: {final_weight:.2f}")
    test = load_hourly(DATA_PATH / "hourly_test.csv", "test")
    test_run_nos = test["Run No"].drop_duplicates().to_numpy()
    X_test = test[FEATS].to_numpy(dtype=np.float32)
    del test
    test_base = fit_first_stage(X, y, X_test)
    probabilities = test_base
    if final_weight > 0:
        meta_indices, meta_probabilities = stack_oof(META_TRAIN_YEARS + [SELECTION_YEAR, CONFIRMATION_YEAR])
        test_refined = fit_second_stage(X[meta_indices], y[meta_indices], meta_probabilities, X_test, test_base)
        probabilities = mix(test_base, test_refined, final_weight)
        del meta_indices, meta_probabilities
    wide = probabilities.reshape(len(test_run_nos), HORIZON, len(UNITS))
    wide = wide.transpose(0, 2, 1).reshape(len(test_run_nos), -1)
    columns = [f"result_committed_{unit}_t{hour}" for unit in UNITS for hour in range(HORIZON)]
    submission = pd.DataFrame(wide, columns=columns)
    submission.insert(0, "Run No", test_run_nos)
    if set(submission.columns) != set(meta["submission_columns"]):
        raise ValueError("Submission columns do not match the dataset metadata.")
    if not np.isfinite(wide).all() or ((wide < 0) | (wide > 1)).any():
        raise ValueError("Submission probabilities must be finite and between zero and one.")
    submission = submission[meta["submission_columns"]]
    submission.to_csv(SUBMISSION_PATH, index=False)
    print(f"Saved {SUBMISSION_PATH}: {submission.shape}")
    display(submission.iloc[:5, :6])
else:
    print("Submission skipped. Set RUN_SUBMISSION = True above when ready.")